In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")
ls_api_key = os.getenv("LANGSMITH_API_KEY")

In [ ]:
import json
from pathlib import Path
from langsmith import Client

# === 1. Initialize the LangSmith client ===
ls_client = Client(api_key=ls_api_key)

# === 2. Reading local QA test JSON file ===
json_path = Path("../qa_dataset.json")
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# === 3. Convert to LangSmith examples format ===
examples = []
for item in data:
    ex = {
        "inputs": {"question": item["question"]},
        "outputs": {"answer": item["answer"]},
        "metadata": {
            "source_id": item.get("source_id", ""),
            "context": item.get("context", "")
        }
    }
    examples.append(ex)

In [ ]:
# === 4. Create a Dataset ===
dataset_name = "FDA QA Dataset_0404_2138"
dataset = ls_client.create_dataset(dataset_name=dataset_name)
print(f"Dataset created: {dataset.id}, name: {dataset.name}")

Dataset created: 23baf39c-622e-4c1d-b6d4-7a14bc64ef68, name: FDA QA Dataset_0404_2138


In [ ]:
# === 5. Upload Examples ===
ls_client.create_examples(dataset_id=dataset.id, examples=examples)
print(f"Uploaded {len(examples)} examples to dataset {dataset.name}")

Uploaded 50 examples to dataset FDA QA Dataset_0404_2138


In [ ]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI

# Grade output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grader_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
        QUESTION: {inputs['question']}
        GROUND TRUTH ANSWER: {reference_outputs['answer']}
        STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

In [29]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool, ..., "Provide the score on whether the answer addresses the question"
    ]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [30]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[
        bool, ..., "Provide the score on if the answer hallucinates from the documents"
    ]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. You will be given FACTS and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. (2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grounded_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc["text"] for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["grounded"]

In [31]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise",
    ]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a set of FACTS provided by the student. Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(
    model="gpt-4o-mini", temperature=0
).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc["text"] for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)
from backend.app.services.rag_service import get_answer
def target_factory(collection_name):
    def target(inputs: dict) -> dict:
        answer_dict = get_answer(inputs["question"], collection_name=collection_name)
        return {"answer": answer_dict["answer"], "documents": answer_dict["documents"]}
    return target

In [10]:
# chunk_sizes = [400, 600, 800]
# chunk_overlaps = [100, 200, 300]
# collections = []
# for chunk_size in chunk_sizes:                 # 改动3
#     for chunk_overlap in chunk_overlaps: 
#         collection_name = f"experiment_chunk{chunk_size}_overlap{chunk_overlap}"
#         collections.append(collection_name)


In [33]:
collections = ["test"]

In [34]:
results = {}
for col in collections:
    print(f"Evaluating collection: {col}")
    target_fn = target_factory(col)
    exp_res = ls_client.evaluate(
        target_fn,
        data=dataset_name,
        evaluators=[correctness, groundedness, relevance, retrieval_relevance],
        experiment_prefix=f"rag_{col}"
    )
    results[col] = exp_res.to_pandas()

Evaluating collection: test
View the evaluation results for experiment: 'rag_test-72318e6f' at:
https://smith.langchain.com/o/4f035ab7-f355-4c59-9d21-0fafb8818fd0/datasets/23baf39c-622e-4c1d-b6d4-7a14bc64ef68/compare?selectedSessions=0fea2c83-8306-4a2c-8a05-1922164434d8




0it [00:00, ?it/s]INFO:httpx:HTTP Request: GET http://localhost:6333/collections/test "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/test/points/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
1it [00:20, 20.75s/it]INFO:httpx:HTTP Request: GET http://localhost:6333/collections/test "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddi

In [36]:
print("\n=== Average Correctness Comparison ===")
print(results)


=== Average Correctness Comparison ===
{'test':                                       inputs.question  \
0   What insights were gained from the root cause ...   
1   How do the vitamin and mineral compositions of...   
2   How can the use of specific biomarkers, such a...   
3   How do the preventive controls requirements of...   
4   How do the evolving classification methods of ...   
5   How does the sensitivity of the screening assa...   
6   How does the testing algorithm for HBsAg and a...   
7   What specific organizational elements should b...   
8   How do the existing advertising and promotion ...   
9   What are the substantive changes in the develo...   
10  How can the training of relevant staff on the ...   
11  How do the AAMI/ANSI/ISO 80369 standards aim t...   
12  How does the FDA's encouragement for laborator...   
13  How do the recommendations for ANDA submission...   
14  How can the challenges of trial recruitment an...   
15  How can understanding matrix effect